# Worksheet 2, Task 3, Track B: the Pizza Ontology as Python Dataclasses – Solution

*Advanced Methods of AI, WiSe 2026/27, University of Bremen*

Track A answers the questions of Task 3 with the OWL reasoner in Protégé. Here you answer the **same questions** with
plain Python dataclasses, using KRROOD's `Symbol` as the base class. Afterwards, compare your answers with a group
that did Track A.

| OWL | Python |
|---|---|
| class (e.g. `CheeseTopping`) | class |
| `SubClassOf` | inheritance |
| object property `hasTopping` | the field `toppings` |
| individual | object |
| defined class (`≡`), classified by the reasoner | ? (that is part (a)) |

Fill in the cells marked **TODO** and run all cells from top to bottom.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List

from krrood.symbol_graph.symbol_graph import Symbol

## The toppings
The topping classes of `pizza.owl` (a selection), with the same names and the same hierarchy.

In [ ]:
# Every class is a dataclass. eq=False: objects are compared by identity (two toppings with the same name
# are still two toppings) and stay hashable, which KRROOD's SymbolGraph needs.
@dataclass(eq=False)
class Food(Symbol):
    name: str


@dataclass(eq=False)
class PizzaTopping(Food): ...


@dataclass(eq=False)
class CheeseTopping(PizzaTopping): ...


@dataclass(eq=False)
class MozzarellaTopping(CheeseTopping): ...


@dataclass(eq=False)
class VegetableTopping(PizzaTopping): ...


@dataclass(eq=False)
class TomatoTopping(VegetableTopping): ...


@dataclass(eq=False)
class OliveTopping(VegetableTopping): ...


@dataclass(eq=False)
class CaperTopping(VegetableTopping): ...


@dataclass(eq=False)
class PepperTopping(VegetableTopping): ...


@dataclass(eq=False)
class HotGreenPepperTopping(PepperTopping): ...


@dataclass(eq=False)
class JalapenoPepperTopping(PepperTopping): ...


@dataclass(eq=False)
class MeatTopping(PizzaTopping): ...


@dataclass(eq=False)
class PeperoniSausageTopping(MeatTopping): ...


@dataclass(eq=False)
class FishTopping(PizzaTopping): ...


@dataclass(eq=False)
class AnchoviesTopping(FishTopping): ...


@dataclass(eq=False)
class FruitTopping(PizzaTopping): ...

## The pizzas

In [ ]:
@dataclass(eq=False)
class Pizza(Food):
    toppings: List[PizzaTopping] = field(default_factory=list)


@dataclass(eq=False)
class NamedPizza(Pizza): ...


@dataclass(eq=False)
class Margherita(NamedPizza): ...


@dataclass(eq=False)
class Napoletana(NamedPizza): ...


@dataclass(eq=False)
class AmericanHot(NamedPizza): ...

## (a) Classify three pizzas

**Predict first:** which of Margherita, Napoletana and AmericanHot are cheesy, vegetarian, interesting or meaty?
Write your prediction into the table on the sheet.

**TODO 1:** create `napoletana_1` and `american_hot_1` with the toppings they have in `pizza.owl`:
* Napoletana: anchovies, capers, mozzarella, olives, tomato
* AmericanHot: hot green peppers, jalapeños, mozzarella, peperoni sausage, tomato

In [ ]:
margherita_1 = Margherita("margherita_1", [MozzarellaTopping("mozzarella_1"), TomatoTopping("tomato_1")])

napoletana_1 = Napoletana("napoletana_1", [AnchoviesTopping("anchovies_1"), CaperTopping("capers_1"),
                                           MozzarellaTopping("mozzarella_2"), OliveTopping("olives_1"),
                                           TomatoTopping("tomato_2")])
american_hot_1 = AmericanHot("american_hot_1", [HotGreenPepperTopping("green_pepper_1"),
                                                JalapenoPepperTopping("jalapeno_1"),
                                                MozzarellaTopping("mozzarella_3"),
                                                PeperoniSausageTopping("peperoni_1"), TomatoTopping("tomato_3")])

In OWL, the four classes below are *defined classes*: the reasoner puts every pizza that meets the definition
into them. A dataclass cannot do that by itself.

**TODO 2:** write the four functions. Each docstring gives the OWL definition.

In [ ]:
def is_cheesy(pizza: Pizza) -> bool:
    """CheesyPizza ≡ Pizza and (hasTopping some CheeseTopping)"""
    return any(isinstance(t, CheeseTopping) for t in pizza.toppings)


def is_vegetarian(pizza: Pizza) -> bool:
    """VegetarianPizza ≡ Pizza and not (hasTopping some MeatTopping) and not (hasTopping some FishTopping)"""
    return not any(isinstance(t, (MeatTopping, FishTopping)) for t in pizza.toppings)


def is_interesting(pizza: Pizza) -> bool:
    """InterestingPizza ≡ Pizza and (hasTopping min 3 Thing)"""
    return len({id(t) for t in pizza.toppings}) >= 3


def is_meaty(pizza: Pizza) -> bool:
    """MeatyPizza ≡ Pizza and (hasTopping some MeatTopping)"""
    return any(isinstance(t, MeatTopping) for t in pizza.toppings)

In [ ]:
def show(pizzas):
    print(f"{'':16}{'cheesy':>9}{'vegetarian':>12}{'interesting':>13}{'meaty':>7}")
    for pizza in pizzas:
        print(f"{pizza.name:16}{is_cheesy(pizza)!s:>9}{is_vegetarian(pizza)!s:>12}"
              f"{is_interesting(pizza)!s:>13}{is_meaty(pizza)!s:>7}")


show([margherita_1, napoletana_1, american_hot_1])

*For the discussion:* does your table agree with the reasoner's? If yes, is it for the same reasons? Look at how the
Margherita is described in `pizza.owl` (slide 4 of the lecture) and at what your `is_vegetarian` checks.

> **Solution.** The table is the same as the reasoner's: all three are cheesy, only Margherita is vegetarian, Napoletana and
AmericanHot are interesting, only AmericanHot is meaty. The reasons differ. The reasoner classifies the Margherita
as vegetarian only because of its closure axiom `hasTopping only (MozzarellaTopping or TomatoTopping)`; here,
`is_vegetarian` is true because the list of toppings is taken to be complete. Every "defined class" had to be
programmed by hand as a function, and the functions classify objects, while the reasoner classified the *classes*
Margherita, Napoletana and AmericanHot. For "at least 3 toppings", OWL needs the topping classes to be disjoint to
know the toppings are different; here they are different objects (counted by `id`, in case the same object is in
the list twice).

## (b) Two impossible classes

In OWL, the reasoner marks these two classes as unsatisfiable (no object can belong to them):
* `CheesyVegetableTopping`: a cheese topping and a vegetable topping,
* `IceCream`: food with toppings (e.g. a fruit topping).

**TODO 3:** define both classes here, create one object of each and see what Python does.

In [ ]:
@dataclass(eq=False)
class CheesyVegetableTopping(CheeseTopping, VegetableTopping): ...


@dataclass(eq=False)
class IceCream(Food):
    toppings: List[PizzaTopping] = field(default_factory=list)


cheesy_vegetable_1 = CheesyVegetableTopping("cheesy_vegetable_1")
ice_cream_1 = IceCream("ice_cream_1", [FruitTopping("strawberries_1")])
print(isinstance(cheesy_vegetable_1, CheeseTopping), isinstance(cheesy_vegetable_1, VegetableTopping),
      isinstance(ice_cream_1, Pizza))

*For the discussion:* why does the reasoner reject them, and why does Python not?

> **Solution.** Both objects are created without complaint. In OWL, `CheeseTopping` and `VegetableTopping` are declared disjoint,
so their intersection is empty; Python classes cannot be declared disjoint, and multiple inheritance makes the
object both. `IceCream` is unsatisfiable in OWL because the *domain* of `hasTopping` is `Pizza`: whatever has a
topping is inferred to be a pizza, and `IceCream` is disjoint from `Pizza`. In Python, `toppings` is just a field;
nothing is concluded from it (`isinstance(ice_cream_1, Pizza)` is `False`). The reasoner finds modelling errors by
reasoning; the dataclasses only find what the program explicitly checks.

## (c) Open world

**TODO 4:** create `unclosed_1`, a pizza of which we only know that it has a mozzarella topping. Is it vegetarian?

In [ ]:
unclosed_1 = Pizza("unclosed_1", [MozzarellaTopping("mozzarella_4")])
show([unclosed_1])

*For the discussion:* the reasoner does **not** classify `UnclosedPizza` as vegetarian. Why does your answer
differ? Could a dataclass express "has a mozzarella topping, and maybe others"?

> **Solution.** `is_vegetarian(unclosed_1)` is `True`: a list says "these are the toppings", so a topping that is not in it does
not exist (closed-world assumption, as in a database). OWL makes the open-world assumption: a meat topping may exist
that nobody stated, so `UnclosedPizza` is not classified as vegetarian until the closure axiom
`hasTopping only MozzarellaTopping` is added. To say "maybe more", the dataclass would need an extra field (e.g.
`toppings_complete: bool`) and functions with a third answer, "unknown". Neither answer is wrong; they rest on
different assumptions. Open versus closed world comes back in KR&R (II), rules over dataclasses (EQL) in Worksheet 3.